# MeteoScreening `G_FF1_0.05_2` (2021-2025) from database (influxdb)

***
**Site**: CH-LAE &nbsp;&nbsp;|&nbsp;&nbsp; **Variable**: `G_FF1_0.05_2` &nbsp;&nbsp;|&nbsp;&nbsp; **Sensor**: Hukseflux **HFP01** soil heat flux plate, forest-floor plot FF1, 0.05 m, logging since 26 Mar 2021 &nbsp;&nbsp;|&nbsp;&nbsp; **Period**: 2021-2025  
**Derived from**: diive notebook template `DatabaseInfluxStepwiseMeteoScreening.ipynb` (version `10`, 2 Sep 2026)  
**Author**: Lukas Hörtnagl (holukas@ethz.ch)

## ℹ️ About this notebook
Download the raw soil heat flux from the InfluxDB database, remove what is unambiguously wrong on the **high-resolution** data, resample to 30MIN, and upload the result back to the database. Screening uses `StepwiseMeteoScreeningDb` from [diive](https://github.com/holukas/diive) (`diive/preprocessing/qaqc/meteoscreening.py`); download and upload use diive's in-house InfluxDB engine (`InfluxIO`, in `diive/core/io/db/influx`).

**Flow:** download (`InfluxIO`) → screen on high-res data (`diive`) → resample to 30MIN → upload.

**Scope is deliberately narrow.** The raw record is ~2.5 million rows, so anything done here is expensive. This notebook takes out only what cannot be a flux at all, and hands over a clean 30MIN series. The plate's calibration, the offset between the two plates, the storage term in the 5 cm of soil above the plate, and any gap handling are settled later in `30_PRODUCTS/`, **on the half-hourly data**, where they cost a fraction as much.

**Outlier detection is stepwise:** run a test, inspect its preview plot, then commit it with `mscr.addflag()`. Re-run with different parameters as often as you like before committing. Run only the tests a variable actually needs. At the end all committed flags are aggregated into one overall quality flag `QCF`.

> **Which sensor this is.** `G_FF1_0.05_2` is one of the **two** Hukseflux **HFP01** plates at the FF1 forest-floor plot, both at 0.05 m. Both were kept across the logger-box rebuild of 24-26 March 2021 - the GIN device records carry them at `CH-LAE_FF1_0.05` before and after - while a third HFP01 at the same location was discarded on 24 March 2021. Acquisition on the new CR1000 started on 26 March 2021, which is where this record begins. The same records place the Campbell 109 thermistor `TS_FF1_0.05_4` directly beside this plate and `TS_FF1_0.05_3` beside plate `_1`, which is what makes them the co-located reference used under *Candidate excursions*.

> ⚠️ **The field name spans two sensor generations, and only the data version separates them.** In `ch-lae_processed`, `G_FF1_0.05_2` under `meteoscreening_mst` holds 2004-09-07 to 2011-12-31 from an **earlier plate set**, and the 2012-2021 era of that older profile is stored under different names again (`G_FF1_0.025_1` to `_3`, ending 2021-03-24). This notebook writes `meteoscreening_diive`, and the delete-before-upload is scoped to that data version, so the older screening is not touched. What must not happen downstream is a merge of the two by name: they are different instruments at different depths with a nine-year hole between them.

> **Database access** needs the `influxdb-client` package, which this project pulls in via the **`diive[db]` extra** (declared in `pyproject.toml`). diive *also* ships a `db` dependency group, but dependency groups are local to the project that declares them - `uv sync --group db` only works inside the diive repo, not from here.

## ⏱️ Timestamp convention (important)
The database stores every timestamp in **UTC** and as **`TIMESTAMP_END`** (the stamp marks the *end* of the averaging interval). Getting this right is the one thing that must not go wrong: the value written back to the database depends on it, and so does every window in `REMOVE_DATES`.

The single knob is `TIMEZONE_OFFSET_TO_UTC_HOURS` (set in *User settings*). It is applied **identically** on download and on upload:

| Stage | Timezone | Convention | Done by |
|---|---|---|---|
| Database | UTC | `TIMESTAMP_END` | InfluxDB |
| After `dbc.download(..., timezone_offset_to_utc_hours=N)` | local (UTC+N) | `TIMESTAMP_END` | InfluxIO |
| During screening | local | `TIMESTAMP_MIDDLE` (converted internally) | `StepwiseMeteoScreeningDb` |
| After `mscr.resample()` | local | back to `TIMESTAMP_END` | diive |
| After `dbc.upload_singlevar(..., timezone_offset_to_utc_hours=N)` | UTC | `TIMESTAMP_END` | InfluxIO |

**Every timestamp printed and plotted below is local time**, one hour ahead of the UTC stamp the database holds. The dates named in the prose of this notebook are local unless they say UTC.

**One place where this bites.** `REMOVE_DATES` is matched against `TIMESTAMP_MID`, not `TIMESTAMP_END`: `StepwiseMeteoScreeningDb` hands the series to `ManualRemoval` *after* the internal conversion, so the record whose `TIMESTAMP_END` is `10:39:00` carries the label `10:38:30` inside the test. A bare `'2025-08-15 15:17:00'` would therefore match **nothing** and the removal would silently do nothing. Any single-minute window added below must be written as the closed interval `[T − 1 min, T]`, which contains exactly the one middle stamp `T − 30 s` and no other. The cell asserts the shape.

## ✏️ User settings (please adjust)

Adjust these before running. What each setting means:

**Site**
- `SITE`, `SITE_LAT`, `SITE_LON`: site ID and coordinates. The coordinates set the day/night split used during screening - not used here, see *Outlier detection*.

**Variable to screen**
- `PROFILE`, `DEPTH`, `REPL`: the plate's position tags. `FIELD` is assembled from them and is the InfluxDB `_field`. **`REPL` is the only thing to change when screening the other plate** - and it changes the co-located thermistor too, so change `NEIGHBOUR_FIELDS` with it.
- `MEASUREMENT`: exactly **one** measurement grouping the variables - `G` for soil heat flux.

**Co-located reference channels**
- `NEIGHBOUR_FIELDS` / `NEIGHBOUR_MEASUREMENTS`: the second plate and both Campbell 109 thermistors. They are downloaded **only for the candidate windows**, are used only to judge whether an excursion is shared, and are never screened, corrected or uploaded here.

**Time range to screen**
- `START`: first timestamp to screen - **is** included.
- `STOP`: upper bound - **is not** included.

**Data settings**
- `TIMEZONE_OFFSET_TO_UTC_HOURS`: the critical timestamp knob - see *Timestamp convention*. Must match how the raw data was logged (`1` for CET winter time) and must be the same value everywhere.
- `DATA_VERSION`: the source data version in the database (`raw`).
- `DIRCONF`: local folder holding the database connection config.

**Resampling**
- `RESAMPLING_FREQ` / `RESAMPLING_AGG`: a heat flux density is a rate, and the half-hourly value is its **mean** over the interval - never `'sum'`, which would report thirty times the flux.

**Physical range**
- `G_MIN`, `G_MAX`: the absolute-limits test, justified from this plate's own measured extremes and checked by the guard cell that follows it.

**Candidate excursions**
- `CANDIDATE_LIMIT`, `CANDIDATE_MAXGAP`, `CANDIDATE_PAD`: the inspection under *Candidate excursions* - which records count as extreme, how far apart two of them still belong to one event, and how much context is downloaded around each event. These produce a **list to judge**, never a removal.

**Parameter help**
- `SHOW_PARAM_HELP`: `True` prints the full docstring of each screening method right before it runs.

In [ ]:
# --- Site ---
SITE = 'ch-lae'
SITE_LAT = 47.478333  # CH-LAE
SITE_LON = 8.364389  # CH-LAE

# --- Variable to screen ---
# The FF1 plot carries two HFP01 plates at 0.05 m, _1 and _2, both logging since
# 26 Mar 2021. REPL selects which one this notebook screens.
PROFILE = 'FF1'  # horizontal position (forest-floor plot)
DEPTH = '0.05'  # plate depth in m
REPL = '2'  # <-- the plate; change NEIGHBOUR_FIELDS with it
FIELD = f'G_{PROFILE}_{DEPTH}_{REPL}'
FIELDS = [FIELD]  # StepwiseMeteoScreeningDb expects a list
MEASUREMENT = 'G'

# --- Co-located reference channels (candidate adjudication only) ---
# The other plate, plus the two Campbell 109 thermistors installed beside the plates
# (TS_FF1_0.05_3 beside plate _1, TS_FF1_0.05_4 beside plate _2). Downloaded only for
# the candidate windows and never screened or uploaded here.
NEIGHBOUR_FIELDS = ['G_FF1_0.05_1', 'TS_FF1_0.05_4', 'TS_FF1_0.05_3']
NEIGHBOUR_MEASUREMENTS = ['G', 'TS']

# --- Time range to screen ---
# Raw resolution is 1MIN throughout, so this range is ~2.5 million records. The plates
# start 2021-03-26 15:22 local; asking for 2021-01-01 simply starts at the first record.
START = '2021-01-01 00:00:01'  # included
STOP = '2026-01-01 00:00:01'  # not included

# --- Data settings ---
DATA_VERSION = 'raw'
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # UTC+01:00 (CET, winter time). Must match how the raw data was logged.
DIRCONF = r'F:\dev\poet\configs'
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'

# --- Resampling ---
RESAMPLING_FREQ = '30min'  # screened high-res data is resampled to this frequency
RESAMPLING_AGG = 'mean'  # (!) a flux density is a rate, never summed

# --- Physical range of soil heat flux at this plate, in W m-2 ---
# Measured on this plate over 2021-2025: -52.96 W m-2 to +127.21 W m-2. The high end is
# real: this plate sits where direct beam reaches the forest floor before leaf-out (see
# 'Candidate excursions'). The limits below sit ~97 W m-2 below the most negative and
# ~23 W m-2 above the most positive value this plate has ever produced, so neither can
# fire on a real flux. They are a garbage guard - a wrong multiplier or a decode error -
# not a spike trim.
# (!) The margin above the maximum is the narrow one. A brighter spring could push a
# sunfleck past +150; re-check the guard cell below rather than assuming it holds.
G_MIN, G_MAX = -150, 150

# --- Candidate excursions (inspection only, never a removal) ---
CANDIDATE_LIMIT = 60  # |G| above this is listed for judgement
CANDIDATE_MAXGAP = '30min'  # two flagged records further apart than this start new events
CANDIDATE_PAD = '1h'  # context downloaded on either side of an event

# --- Parameter help ---
SHOW_PARAM_HELP = False

## 🤖 Auto settings

### Buckets (do not adjust)

In [ ]:
BUCKET_RAW = f'{SITE}_raw'  # source bucket, e.g. 'ch-lae_raw'
BUCKET_PROCESSED = f'{SITE}_processed'  # destination bucket, e.g. 'ch-lae_processed'
print(f'Screening variable:             {FIELD}')
print(f'Source bucket (raw data):       {BUCKET_RAW}')
print(f'Destination bucket (processed): {BUCKET_PROCESSED}')

### Imports

In [ ]:
import importlib.metadata
import warnings
from datetime import datetime

import pandas as pd

import diive as dv
from diive.core.io.db.influx import InfluxIO  # needs influxdb-client, via the diive[db] extra

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive v{importlib.metadata.version('diive')}")

## ⬇️ Download data from database

### Connect to database

In [ ]:
dbc = InfluxIO(dirconf=DIRCONF)

Optional - list all fields available in the measurement (does not check the selected time range). Worth a look here because `G` also holds the **older forest-floor plates** (`G_M1_0.05_1`, `G_M2_0.05_1`, `G_M3_0.05_1`), which are 10MIN, end at the March 2021 logger rebuild, and are different instruments at a different spot. They must never be spliced onto this record:

In [ ]:
display(dbc.show_fields_in_measurement(bucket=BUCKET_RAW, measurement=MEASUREMENT))

### Download
Returns three objects:
- `data_simple`: high-res time series, one column per variable (nice to look at).
- `data_detailed`: dict `{varname: DataFrame}` with each variable's time series **and its database tags** - this is what the screening consumes.
- `assigned_measurements`: the auto-detected measurement per variable (a sanity check).

In [ ]:
%%time
data_simple, data_detailed, assigned_measurements = dbc.download(
    bucket=BUCKET_RAW,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
)

### Inspect downloaded data

In [ ]:
data_simple

In [ ]:
assigned_measurements

Drop any requested variable that has no data in this period:

In [ ]:
vars_not_available = [v for v in FIELDS if v not in data_detailed.keys()]
for rem in vars_not_available:
    FIELDS.remove(rem)
    print(f'Removed {rem} from FIELDS (no data in this period).')
print(f'Data available for: {list(data_detailed.keys())}')
assert FIELD in data_detailed, f'(!) {FIELD} returned no data - nothing to screen'

### Verify download timestamps and time resolution
Confirm the timestamps look right: **local time** (UTC+`TIMEZONE_OFFSET_TO_UTC_HOURS`), marking the **end** of each averaging interval. The database itself stores UTC - `InfluxIO` applied the offset on download. Eyeball the first/last stamps against the `START`/`STOP` you requested.

The second cell states something this record has and the neighbouring soil variables do **not**: a single raw time resolution. `SWC` and `TS` at this plot span a 10MIN era before the March 2021 rebuild and a 1MIN era after, and diive upsamples the coarse era onto the fine grid, which degenerates every difference-based test there. The plates only ever logged on the rebuilt logger, so their record is 1MIN throughout and no such split exists. The assertion is what keeps that claim true if the record is ever re-ingested.

In [ ]:
for v in data_detailed.keys():
    idx = data_detailed[v].index
    print(f'{v}: index name={idx.name!r}, tz={idx.tz}, freq={idx.freqstr}')
    print(f'   first={idx[0]}   last={idx[-1]}')
print(f'\nApplied UTC offset: +{TIMEZONE_OFFSET_TO_UTC_HOURS}h (timestamps above are local time)')

In [ ]:
# One raw resolution only: the modal record spacing must be 1 minute, and gaps aside
# it must dominate. A second era would show up as a second large entry here.
_spacing = data_detailed[FIELD].index.to_series().diff().value_counts()
print(_spacing.head(10).to_string())
_share = _spacing.iloc[0] / _spacing.sum()
print(f'\nModal spacing {_spacing.index[0]} covers {_share:.4%} of all record gaps.')
assert _spacing.index[0] == pd.Timedelta('1min'), '(!) raw resolution is not 1MIN'
assert _share > 0.99, '(!) more than one raw time resolution in this record - see the note above'

### Plot downloaded high-res data

In [ ]:
for varname, frame in data_detailed.items():
    dv.plotting.TimeSeries(series=frame[varname]).plot()

## ▶️ Start MeteoScreening with `diive`

In [ ]:
mscr = dv.qaqc.StepwiseMeteoScreeningDb(
    site=SITE,
    data_detailed=data_detailed,
    fields=FIELDS,
    site_lat=SITE_LAT,
    site_lon=SITE_LON,
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
)
mscr.showplot_orig()

## 🔍 Outlier detection
Run a test → inspect its preview → commit with `mscr.addflag()`. Only the committed flag of the **most recent** test is added. Skip any test a variable does not need.

**Two tests are committed here** and nothing else: **absolute limits** and the **missing-values** flag. Manual removal is present, guarded, and empty by default. Why the rest are off:

- **Never split day and night.** Soil heat flux does have a strong diurnal cycle, but none of the tests run here is a distribution test, so a solar-geometry split would only halve the sample behind every threshold. Every test below runs with `separate_day_night=False`.
- **Distribution-wide tests measure the canopy, not a fault.** The forest floor of a deciduous stand receives direct beam before leaf-out and almost none after, so the same +90 W m-2 is ordinary in April and impossible in July. A z-score against the multi-year mean flags the spring.
- **Differencing-based tests flag the real signal first.** A sunfleck arriving at the plate is a genuine step of tens of W m-2 from one minute to the next; that is exactly the shape a Hampel filter on differences or an increments z-score is built to remove. Both were left off for that reason, not because the record is quiet.
- **What is left is what absolute limits can do**: reject values that are not a soil heat flux at all. On the current record it flags nothing, which is the honest result and is reported as such.

> **The keyword changed in diive `v0.91.0`.** The day/night switch is `separate_day_night` everywhere; the template's `separate_daytime_nighttime` raises. Every commented call below is already written in the current form.

In [ ]:
mscr.start_outlier_detection()

Plot the current cleaned data at any point during detection:

In [ ]:
for key, val in mscr.outlier_detection.items():
    val.showplot_cleaned(interactive=False)

### Candidate excursions
No automatic test can separate a sunfleck from a disturbed plate, because both are large, short and abrupt. What separates them is whether the **neighbours moved too**: this plot carries a second HFP01 a few metres away and a Campbell 109 thermistor beside each plate, so an event shared by four channels on two measurement principles is the forest floor, and an event on one plate alone is that plate.

The two cells below group every record beyond `CANDIDATE_LIMIT` into contiguous events and print each one next to its neighbours. **They remove nothing.** They produce the list a person judges before writing anything into `REMOVE_DATES` - the same rule the `SWC` notebooks follow for their profile cross-check.

**The excursions of this plate are sunflecks, not faults.** At `CANDIDATE_LIMIT = 60` this plate has tens of events where plate `_1` has two, and they carry the signature of direct beam reaching the forest floor rather than of an instrument:

- they are almost all **positive** (heat into the soil), which is what sunlight on the litter layer does and what an electrical fault has no reason to prefer;
- they are **short**, a few minutes to about half an hour;
- they **recur on consecutive days at nearly the same clock time**, and that time drifts with the season (about 10:40 local in mid-April, about 09:35 in late May, about 14:45 in mid-June) - a sun path, not a schedule any logger keeps;
- they **stop being this large once the canopy closes**, which is why a distribution-wide test over the whole record would flag the spring rather than a fault.

So `REMOVE_DATES` is empty for this plate, and the absolute limits are set wide enough to let +127 W m-2 through. The one candidate in the pair sits on plate `_1` (2021-04-12) and is written up there. Sunflecks are also the reason the resampled half hours of this plate deserve a second look downstream: see *Resampling*.

In [ ]:
def candidate_events(series, limit=CANDIDATE_LIMIT, maxgap=CANDIDATE_MAXGAP):
    """Group the records beyond +/-limit into contiguous excursions.

    Two flagged records further apart than *maxgap* start separate events, so a
    sunfleck lasting half an hour is one row and not two hundred.
    """
    hits = series[series.abs() > limit].dropna()
    cols = ['start', 'end', 'duration', 'n', 'min', 'max']
    if hits.empty:
        return pd.DataFrame(columns=cols)
    grp = hits.index.to_series().diff().gt(pd.Timedelta(maxgap)).cumsum().values
    g = hits.groupby(grp)
    ev = pd.DataFrame({
        'start': g.apply(lambda s: s.index[0]),
        'end': g.apply(lambda s: s.index[-1]),
        'n': g.size(),
        'min': g.min().round(2),
        'max': g.max().round(2),
    })
    ev['duration'] = ev['end'] - ev['start']
    return ev[cols].reset_index(drop=True)


# Negative control: a helper that silently returns nothing is worse than one that fails.
_probe = pd.Series([0.0, 0.0, 999.0, 0.0],
                   index=pd.date_range('2021-06-01', periods=4, freq='1min'))
assert len(candidate_events(_probe, limit=100)) == 1, '(!) grouping missed a lone spike'
assert candidate_events(_probe, limit=1000).empty, '(!) grouping invented an event'

_series = data_detailed[FIELD][FIELD]
events = candidate_events(_series)
print(f'{FIELD}: {int(_series.abs().gt(CANDIDATE_LIMIT).sum())} records beyond '
      f'+/-{CANDIDATE_LIMIT} W m-2, grouped into {len(events)} events '
      f'(local timestamps, TIMESTAMP_END):')
display(events)

In [ ]:
def event_context(event, pad=CANDIDATE_PAD, agg='5min'):
    """This plate and its neighbours around one candidate event, at reduced resolution.

    Downloads only the event window plus *pad* on either side - a full-period download of
    three more 1MIN channels would cost as much again as the screening itself.
    """
    start = (event['start'] - pd.Timedelta(pad)).strftime('%Y-%m-%d %H:%M:%S')
    stop = (event['end'] + pd.Timedelta(pad)).strftime('%Y-%m-%d %H:%M:%S')
    nb_simple, _, _ = dbc.download(
        bucket=BUCKET_RAW,
        measurements=NEIGHBOUR_MEASUREMENTS,
        fields=NEIGHBOUR_FIELDS,
        start=start,
        stop=stop,
        timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
        data_version=DATA_VERSION,
    )
    ctx = pd.concat([_series.loc[start:stop], nb_simple], axis=1)
    return ctx.resample(agg).mean().round(2)


# Inspect the events one by one. Set MAX_EVENTS higher to walk through all of them; the
# plates that see sunflecks have tens, and reading three of them is usually enough to
# recognise the pattern.
MAX_EVENTS = 3
for _, _ev in events.head(MAX_EVENTS).iterrows():
    print(f"\n=== {FIELD}: {_ev['start']} to {_ev['end']} "
          f"({_ev['n']} records, {_ev['min']} to {_ev['max']} W m-2) ===")
    print(event_context(_ev).to_string())

### Manual removal
Flag specific timestamps or time ranges for removal - known sensor failures, maintenance windows, logger artefacts. Give `[start, stop]` pairs and/or single timestamps.

**`REMOVE_DATES` is empty for this plate**, and the cells below skip themselves when it is. Nothing in this record has been shown to be an artefact: the candidates are written up above with what would settle each of them. An empty list is a statement, not an omission - it says no window here deletes data on anyone's say-so.

> ⚠️ **A removal window belongs to a sensor, not to a variable.** Do not carry a window from the other plate, from the `TS` notebooks, or from the shared 2012 tower faults into this list. Each must be re-derived from this plate's own record, and this notebook must then grow the two-direction audit that a non-empty `REMOVE_DATES` requires: that the window is genuinely anomalous against the neighbours (re-computed here, not quoted from the prose), and that nothing real was lost with it.

> **Write single minutes as `[T − 1 min, T]`** - see *Timestamp convention*. `ManualRemoval` runs on `TIMESTAMP_MID`, so the closed interval selects exactly the one record whose `TIMESTAMP_END` is `T`, and a bare timestamp would select nothing. The shape is asserted below.

In [ ]:
if SHOW_PARAM_HELP:
    help(dv.outliers.ManualRemoval)

In [ ]:
# Empty on purpose - see the note above and 'Candidate excursions'.
REMOVE_DATES = []

# A window that brackets no middle stamp removes nothing at all, silently - so check the
# shape of every entry. A bare timestamp is the form that fails this way, so it is
# rejected outright rather than skipped over.
def check_removal_window(w):
    """Raise unless *w* is a [start, stop] pair that can select a record."""
    assert isinstance(w, (list, tuple)), (
        f'(!) {w!r} is a bare timestamp; ManualRemoval runs on TIMESTAMP_MID, so write '
        f'a single minute as the closed bracket [T - 1 min, T]')
    assert len(w) == 2, f'(!) window {w} is not a [start, stop] pair'
    _s, _e = pd.Timestamp(w[0]), pd.Timestamp(w[1])
    assert _e > _s, f'(!) window {w} does not run forwards'
    if _e - _s < pd.Timedelta('2min'):
        assert _e - _s == pd.Timedelta('1min'), (
            f'(!) window {w} is neither a one-minute bracket nor a longer period')


# Negative control: the guard is never exercised while REMOVE_DATES is empty, so prove
# here that it rejects the two shapes that would silently remove nothing.
for _bad in ['2025-08-15 15:17:00', ['2025-08-15 15:17:00', '2025-08-15 15:17:30']]:
    try:
        check_removal_window(_bad)
    except AssertionError:
        pass
    else:
        raise AssertionError(f'(!) the window guard accepted {_bad!r}')
check_removal_window(['2025-08-15 15:16:00', '2025-08-15 15:17:00'])  # must not raise

for _w in REMOVE_DATES:
    check_removal_window(_w)

if REMOVE_DATES:
    mscr.flag_manualremoval_test(remove_dates=REMOVE_DATES, showplot=True, verbose=True)
else:
    print('No manual removal windows for this plate - skip the addflag cell below.')

In [ ]:
if REMOVE_DATES:
    mscr.addflag()

### Absolute limits
Flags values outside a fixed physical range `[minval, maxval]`. This is the one automatic test this variable gets, and its job is narrow: reject anything that is not a soil heat flux at all - a wrong multiplier, a decode error, a disconnected differential channel - while letting every real sunfleck and every real cold-rain excursion through.

The guard cell first re-derives the record's own extremes and its 0.01-99.99 % band and checks that the limits sit well outside them. A limit inside the measurement's own ordinary range is mis-set, not strict.

> ⚠️ **Do not clip instead.** `correction_setto_max_threshold(threshold=150)` would turn a decode error into a fabricated 150 W m-2 - a value inside the physical range that nothing downstream could identify as false. A removed value is honest; a clipped one is not.

In [ ]:
if SHOW_PARAM_HELP:
    help(dv.outliers.AbsoluteLimits)

In [ ]:
# The limits must sit outside what this plate actually measures, by a margin wider than
# its own extreme-but-real excursions. Re-derived on every run, so a re-ingested or
# extended record cannot leave the justification standing while the data move.
_series = data_detailed[FIELD][FIELD]  # re-derived here so this cell stands alone
_lo, _hi = _series.quantile([0.0001, 0.9999])
print(f'{FIELD} over {START[:10]} to {STOP[:10]}:')
print(f'  measured extremes: {_series.min():.2f} to {_series.max():.2f} W m-2')
print(f'  0.01-99.99 % band: {_lo:.2f} to {_hi:.2f} W m-2')
print(f'  absolute limits:   {G_MIN} to {G_MAX} W m-2')
print(f'  margin below/above the band: {_lo - G_MIN:.2f} / {G_MAX - _hi:.2f} W m-2')
assert G_MIN < _lo - 20 and G_MAX > _hi + 20, (
    '(!) the absolute limits sit within 20 W m-2 of the ordinary range of this record - '
    'they would fire on real fluxes')

# The band check alone is not enough. At ~2.5 million records the 0.01 % tail holds a few
# hundred of them, so a run of real sunflecks could sit beyond G_MAX while the quantile
# barely moves - and the test would then delete measurements, which the section above
# promises it never does. Assert on the extremes, which is the claim actually being made.
_flagged = int(((_series < G_MIN) | (_series > G_MAX)).sum())
print(f'\nRecords the test will flag: {_flagged}')
assert _flagged == 0, (
    f'(!) absolute limits [{G_MIN}, {G_MAX}] would flag {_flagged} measured records. '
    f'Decide whether those are garbage or a real excursion before letting this test '
    f'remove them - and do not widen the limits merely to silence this assertion.')

In [ ]:
mscr.flag_outliers_abslim_test(minval=G_MIN, maxval=G_MAX, showplot=True, verbose=True)

In [ ]:
mscr.addflag()

### Other tests (all off)
Left switched off for the reasons given at the top of *Outlier detection*. If you do enable one, judge it against the **canopy season** rather than the whole record: a threshold tuned on the closed-canopy summer will remove the spring sunflecks wholesale. Note also that differencing-based tests compute their differences **after dropping missing records**, so the records flanking every gap are compared across the gap and look like steps.

In [ ]:
# Optional, all off by default - read the note above before enabling any of these.
# Keyword names follow the diive v0.91.0 API (separate_day_night, not separate_daytime_nighttime).
# mscr.flag_outliers_hampel_test(window_length=60 * 24, n_sigma=8, use_differencing=False,
#                                separate_day_night=False,
#                                repeat=False, showplot=True, verbose=True)
# mscr.flag_outliers_zscore_test(thres_zscore=4.5, separate_day_night=False,
#                                repeat=True, showplot=True, verbose=True)
# mscr.flag_outliers_zscore_rolling_test(thres_zscore=4.5, winsize=60 * 24 * 7,
#                                        repeat=True, showplot=True, verbose=True)
# mscr.flag_outliers_localsd_test(n_sd=7, winsize=60 * 24 * 7, constant_sd=False,
#                                 separate_day_night=False,
#                                 repeat=False, showplot=True, verbose=True)
# mscr.flag_outliers_increments_zcore_test(thres_zscore=40, repeat=True, showplot=True, verbose=True)
# mscr.addflag()

### Missing values
Not an outlier test - flags missing records so they are counted in the overall `QCF`.

In [ ]:
mscr.flag_missingvals_test(verbose=True)

### Overall quality flag QCF
Aggregate all committed test flags into one overall flag `QCF` (0 = good, 1 = marginal, 2 = bad) and filter the series. Required before corrections and resampling.

In [ ]:
mscr.finalize_outlier_detection()

#### Reports

In [ ]:
mscr.report_outlier_detection_qcf_evolution()

In [ ]:
mscr.report_outlier_detection_qcf_flags()

In [ ]:
mscr.report_outlier_detection_qcf_series()

#### Plots

In [ ]:
mscr.showplot_outlier_detection_qcf_heatmaps()
# mscr.showplot_outlier_detection_qcf_timeseries()

## 🔧 Corrections
Applied to the high-res, QCF-filtered data. **None of them apply to a soil heat flux plate**, and every call below stays commented out. The corrections diive offers are offset removals for radiation and relative humidity, threshold clipping, and setting ranges to a constant - every one of them writes a value the sensor did not measure.

> ⚠️ **The nighttime zero-offset correction is not for this variable.** It exists for upward-facing radiometers, whose true nighttime value is zero. A buried plate's nighttime value is **not** zero - it is the night-time upward flux, which is the larger half of the daily cycle in a shaded forest floor. Applying it would delete the signal.

> ⚠️ **Do not set the exact value 0 to missing.** `correction_set_exact_value_to_missing(values=[0])` catches stuck sensors on other variables; here it would delete the two zero crossings this series makes every day, which are the most ordinary values it takes.

> **The plate's own systematic questions belong downstream.** The offset between the two plates, the HFP01's under-reading in soil of differing conductivity, and the heat stored in the 5 cm of soil above the plate (which decides whether this series can be used in an energy-balance closure at all) are all corrections that need `TS` and `SWC` beside them. They are settled in `30_PRODUCTS/`, on the 30MIN data.

In [ ]:
mscr.showplot_cleaned()

Inspect the most frequent values first (read-only, safe to run). A plate flux is a continuous float, so this series should show **almost no repeated values**: a handful of distinct values carrying a large share of the record would mean a stuck channel, and a single value repeating far more often than its neighbours would mean a sentinel written by the logger.

In [ ]:
for ff in mscr.fields:
    vc = mscr.series_hires_cleaned[ff].value_counts()
    print(f'--- {ff} ({mscr.series_hires_cleaned[ff].count():,} records, '
          f'{mscr.series_hires_cleaned[ff].nunique():,} distinct) ---')
    print(vc.head(20))

In [ ]:
# All commented out on purpose - none of these apply to a soil heat flux plate.
# mscr.correction_remove_nighttime_zero_offset()
# mscr.correction_setto_max_threshold(threshold=150)
# mscr.correction_setto_min_threshold(threshold=-150)
# mscr.correction_setto_value(dates=[['2021-04-01', '2021-04-05']], value=0, verbose=1)
# mscr.correction_set_exact_value_to_missing(values=[0])
# mscr.showplot_cleaned(interactive=False)

## 🔁 Resampling

### Resample to 30MIN
Resample the screened high-res series to `RESAMPLING_FREQ`. The output timestamp is `TIMESTAMP_END` again (see *Timestamp convention*), ready for upload. **This is the handover point**: everything downstream of here works on the half-hourly series.

In [ ]:
# mincounts_perc stays at the template default of .25, i.e. 7.5 of the 30 minutes a full
# half hour holds. That is a low bar for a mean, and for this variable it is a lower bar
# than it looks: under a sunfleck the flux is not smooth within the half hour, so a half
# hour reconstructed from a quarter of its minutes can land well above or below the mean
# of the full interval depending on which minutes survived. The screening here removes
# almost nothing, so in practice a partial half hour means the logger was down - but the
# per-interval count is worth carrying into 30_PRODUCTS/ rather than assuming it away.
mscr.resample(to_freqstr=RESAMPLING_FREQ, agg=RESAMPLING_AGG, mincounts_perc=.25)
mscr.showplot_resampled()

### Check the resampled time resolution

In [ ]:
for v in mscr.resampled_detailed.keys():
    freq = dv.times.DetectFrequency(index=mscr.resampled_detailed[v].index, verbose=True).get()
    status = 'PASSED' if freq == RESAMPLING_FREQ else '(!) FAILED'
    print(f'{status} - {v}: {freq}')

## ⬆️ Upload data to database

**Re-uploading overwrites the same variant - safe to re-run.** With `delete_from_db_before_upload=True` (below), the upload first *deletes*, then writes. The delete is scoped to the exact match `_measurement` + `varname` + `data_version` (`meteoscreening_diive`) over the uploaded time range, so re-screening a period replaces only its previous screened result. It never touches the raw data (different `data_version`, and a different `_raw` bucket), other variables, or other data versions. The delete-first step (rather than a plain overwrite) matters because InfluxDB keys a point by its full tag set: if a tag changed between runs (e.g. `units`, `gain`, `offset`), a plain write would leave the old point as a **duplicate** - the delete removes it regardless of tags.

> **The scoping is what protects the older screening here.** `G_FF1_0.05_2` already exists in `ch-lae_processed` under `meteoscreening_mst`, holding 2004-2011 from an earlier plate set. That is a different data version, so this upload leaves it untouched - and the time ranges do not overlap either. Both facts are worth re-checking if the delete is ever widened.

In [ ]:
print(f'Uploading to bucket {BUCKET_PROCESSED}')
for v in mscr.resampled_detailed.keys():
    dbc.upload_singlevar(
        to_bucket=BUCKET_PROCESSED,
        to_measurement=assigned_measurements[v],
        var_df=mscr.resampled_detailed[v],
        timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
        delete_from_db_before_upload=True,
    )

### Verify upload
Download the just-uploaded data back and confirm the time resolution and timestamps. The offset is the same `TIMEZONE_OFFSET_TO_UTC_HOURS`, so the timestamps below should again be local `TIMESTAMP_END` - matching what you screened.

In [ ]:
# Fresh variable names so the screened originals (data_detailed etc.) are not overwritten:
check_simple, check_detailed, check_measurements = dbc.download(
    bucket=BUCKET_PROCESSED,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version='meteoscreening_diive',
)
check_simple

In [ ]:
for v in check_detailed.keys():
    idx = check_detailed[v].index
    freq = dv.times.DetectFrequency(index=idx, verbose=True).get()
    status = 'PASSED' if freq == RESAMPLING_FREQ else '(!) FAILED'
    print(f'{status} - {v}: freq={freq}, first={idx[0]}, last={idx[-1]}')

## ✅ End of notebook

In [ ]:
_end = datetime.now()
print(f"Finished: {_end.strftime('%Y-%m-%d %H:%M:%S')}  "
      f"(runtime {str(_end - NOTEBOOK_START).split('.')[0]})")

***
### 📝 Notes for this variable

- **Mean, not sum.** `RESAMPLING_AGG = 'mean'` - a heat flux density is a rate, and the half-hourly value is its average over the interval.
- **Units are W m-2**, `gain 1.0`, `offset 0.0`, filegroup `12_meteo_forestfloor`, raw source `G_FF1_0x05_2_Avg` in the 1MIN TOA5 table. No unit break anywhere in this record.
- **Sensor identity.** A Hukseflux **HFP01** at 0.05 m in the FF1 forest-floor plot, one of two kept across the logger-box rebuild of 24-26 March 2021; a third plate at the same location was discarded then. Identity and dates come from the GIN device records, which also place `TS_FF1_0.05_4` directly beside this plate.
- **One raw time resolution.** 1MIN throughout, because the plates only ever logged on the rebuilt CR1000. This is the exception among the forest-floor variables: `SWC` and `TS` at this plot span a 10MIN and a 1MIN era, and every difference-based test degenerates on their upsampled early half. Nothing of that applies here, and the assertion after the download keeps the claim honest.
- **The record starts 2021-03-26 15:22 local**, when acquisition on the new logger began. The first quarter of 2021 is empty by construction, not by fault.
- **Extreme values are mostly weather.** Before leaf-out, direct beam reaches the forest floor and drives the plates well beyond their summer range; cold rain reaching the litter drives them sharply negative for a few minutes. Both are real and both are kept. The *Candidate excursions* section is what tells them apart from a disturbed plate, and it produces candidates only.
- **A name is not a continuity.** `G_FF1_0.05_2` in `ch-lae_processed` also holds 2004-2011 under `meteoscreening_mst`, from an earlier plate set, and the 2012-2021 era of that older profile is stored as `G_FF1_0.025_1` to `_3`. Three eras, two depths, two screenings, and a nine-year hole between the second and the third. Whatever `30_PRODUCTS/` exports must carry a `SOURCE` flag naming the sensor generation rather than splicing them.
- **Deferred to `30_PRODUCTS/`, on the 30MIN data:** the offset between the two plates, the storage term in the soil above them, whether the product exports the plates individually or their mean (the FLUXNET convention is `G_1_1_1`, `G_2_1_1`, …), and any gap handling. This notebook exports gaps where the logger was down and does not fill them.

### ♻️ Reusing this notebook
Copy to `G_FF1_0.05_<repl>_2021-2025.ipynb` and change **`REPL`** in *User settings*. Then:

1. **Change `NEIGHBOUR_FIELDS` with it.** The co-located thermistor differs per plate (`TS_FF1_0.05_3` beside plate `_1`, `TS_FF1_0.05_4` beside plate `_2`), and the other plate is the other one. Leaving the target itself in `NEIGHBOUR_FIELDS` would compare the series with itself and make every event look shared.
2. **Re-derive `REMOVE_DATES` from that plate's own record.** Both plates are empty today, and the one candidate in the pair sits on plate `_1`. A window belongs to a **sensor**: the plates are metres apart and see different sun, different litter and different water.
3. **Re-run the absolute-limits guard rather than carrying the numbers.** `[-150, 150]` covers both plates today, but plate `_2` reaches +127 W m-2 and its margin above the record is the narrow one. The guard cell re-derives the band on every run and fails if a limit moves inside it.
4. **Do not carry the two plates' prose across.** The adjudication under *Candidate excursions* describes what that plate's own excursions were found to be, and the two plates differ: one is dominated by sunflecks, the other has two isolated events.